# exploracao_gold_dec_fec

Conferências do `gold_dec_fec` depois do checkup de 23/10. Lê `silver.tb_ocorrencias_rede` e as Gold `agg_dec_fec_mensal` e `agg_kpis_mensal`. **Não grava nada.**

- **Rode depois do Job** (precisa da Silver e das Gold `gold_dec_fec` e `gold_kpis`).
- As decisões do checkup (OSCILACAO fora, dia crítico, duração efetiva, Cc por mês, ocorrências antes da ligação e flags zeradas) viraram regra ou checagem na `silver_rede_atendimento` e no `gold_dec_fec`. As consultas que serviram só para tomá-las saíram daqui; continuam no histórico do Git.

| Seção | Pergunta |
|---|---|
| 1 | Quanto cada tipo de ocorrência pesa na Silver? |
| 2 | Do que é feito julho de 2025? |
| 3 | Quantas interrupções viram o mês? |
| 4 | O DEC/FEC do `gold_dec_fec` bate com o do `gold_kpis`? |

In [0]:
%run ../bronze-to-silver/funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")

OC = f"{CATALOGO}.silver.tb_ocorrencias_rede"
G = f"{CATALOGO}.gold"
sql = lambda q: display(spark.sql(q))
print(f"ocorrências: {spark.table(OC).count():,}")

## 1. Peso de cada tipo de ocorrência
OSCILACAO não entra no DEC/FEC (`flag_sem_interrupcao`): é variação de tensão, a energia não falta. **Olhe:** quanto cada tipo pesa nas ocorrências e nas horas da Silver.

In [0]:
sql(f'''
SELECT tipo, COUNT(*) AS interrupcoes, ROUND(SUM(duracao_min) / 60, 0) AS horas,
       ROUND(100 * SUM(duracao_min) / SUM(SUM(duracao_min)) OVER (), 1) AS pct_horas
FROM {OC} GROUP BY tipo ORDER BY interrupcoes DESC
''')

## 2. Composição de julho de 2025
Julho de 2025 tem os 31 dias críticos (`flag_dia_critico`) e 58% do DEC do P2. **Olhe:** se o mês é um evento só (uma causa e um tipo) ou uma mistura.

In [0]:
sql(f'''
SELECT causa, tipo, COUNT(*) AS n, ROUND(SUM(duracao_min) / 60) AS horas
FROM {OC} WHERE date_format(data_abertura, 'yyyy-MM') = '2025-07'
GROUP BY ALL ORDER BY n DESC
''')

## 3. Interrupções que viram o mês
A interrupção fica inteira no mês de abertura (regra combinada). **Olhe:** quantas horas iriam para o mês seguinte.

In [0]:
sql(f'''
SELECT date_format(data_abertura, 'yyyy-MM') AS mes_abertura, COUNT(*) AS n,
       ROUND(SUM(duracao_min) / 60) AS horas_no_mes_de_abertura
FROM {OC}
WHERE date_format(data_abertura, 'yyyy-MM') <> date_format(data_encerramento, 'yyyy-MM')
GROUP BY 1 ORDER BY n DESC
''')

## 4. gold_dec_fec x gold_kpis
Os dois devem dar o mesmo DEC/FEC global, com e sem dia crítico. A soma dos alimentadores divide pelo Cc de cada mês (Σ `ucs_atendidas`). **Olhe:** se aparece alguma linha; vazio = batem.

In [0]:
sql(f'''
WITH dec_fec AS (
  SELECT mes_referencia,
         SUM(horas_interrupcao) / SUM(ucs_atendidas) AS dec,
         SUM(interrupcoes) / SUM(ucs_atendidas) AS fec,
         SUM(dec_sem_dia_critico * ucs_atendidas) / SUM(ucs_atendidas) AS dec_sem_dc
  FROM {G}.agg_dec_fec_mensal GROUP BY 1)
SELECT k.mes_referencia, k.dec AS dec_kpis, ROUND(d.dec, 4) AS dec_dec_fec, k.fec AS fec_kpis, ROUND(d.fec, 4) AS fec_dec_fec,
       k.dec_sem_dia_critico AS dec_sem_dc_kpis, ROUND(d.dec_sem_dc, 4) AS dec_sem_dc_dec_fec
FROM {G}.agg_kpis_mensal k JOIN dec_fec d USING (mes_referencia)
WHERE ABS(k.dec - d.dec) > 0.0001 OR ABS(k.fec - d.fec) > 0.0001
   OR ABS(k.dec_sem_dia_critico - d.dec_sem_dc) > 0.0002   -- soma de valores já arredondados por alimentador
ORDER BY 1
''')